# Milestone 1 — Parkinson's Disease Target Ranking

**Context.md §36.** A transparent, non-ML target ranking built only from Open Targets evidence.

> Scores here are *prioritization hypotheses*, not validated findings. A high score does not
> mean a target will yield an effective drug. See `docs/limitations.md`.

The logic lives in the package (Context.md §34 — notebooks are for exploration and
visualization, not for holding the pipeline). This notebook is a thin layer over it, so
anything you learn here is reproducible from the command line:

```bash
uv run python scripts/run_milestone1.py
```

In [ ]:
import polars as pl

from target_prioritization.milestone1 import (
    KNOWN_PARKINSONS_GENES,
    ablation_movement,
    run_milestone_1,
)
from target_prioritization.models.baseline import SCORE_COLUMN

pl.Config.set_tbl_rows(25)
result = run_milestone_1(write_outputs=False)
result.disease.name, result.disease.efo_id, result.ranked.height

## 1. What the candidate set looks like

Candidate generation is deliberately simple (Context.md §13): every target Open Targets
associates with the disease. The interesting property is how *thin* the evidence is.

In [ ]:
evidence_profile = (
    result.ranked.group_by("n_evidence_types")
    .len()
    .sort("n_evidence_types")
    .rename({"len": "n_targets"})
)
evidence_profile

In [ ]:
# The single most important fact about this dataset.
lit_only = result.ranked.filter(
    (pl.col("n_evidence_types") == 1) & pl.col("dim__literature").is_not_null()
)
print(
    f"{lit_only.height:,} of {result.ranked.height:,} candidates ({100 * lit_only.height / result.ranked.height:.0f}%) have literature evidence and nothing else."
)

That is why `literature` carries a low weight and why `evidence_diversity` is scored at all.
A score dominated by publication volume would rank genes by fame (Context.md §32.2).

## 2. The ranking

Contributions sum exactly to the score, so the table below is fully decomposable —
no SHAP required.

In [ ]:
top20 = result.ranked.head(20).select(
    [
        "rank",
        "gene_symbol",
        SCORE_COLUMN,
        "dim__genetics",
        "n_evidence_types",
        "dim__functional",
        "dim__literature",
        "dim__druggability",
        "evidence_completeness",
    ]
)
top20

## 3. Does it recover known biology?

The one question this milestone exists to answer. These genes are **not labels** —
nothing in the scoring knows about them.

In [ ]:
known = pl.DataFrame(
    {
        "gene_symbol": list(KNOWN_PARKINSONS_GENES.values()),
        "rank": [result.known_gene_ranks[s] for s in KNOWN_PARKINSONS_GENES.values()],
    }
).sort("rank")
print(known)
print(f"\nAcceptance check passed: {result.acceptance_passed}")

## 4. Why did LRRK2 rank first?

The contributions decompose the score exactly.

In [ ]:
explanation = result.explanations[0]
print(f"{explanation.gene_symbol}  score={explanation.score:.4f}")
print(f"evidence completeness: {explanation.evidence_completeness:.0%}")
print(f"missing dimensions: {explanation.missing_dimensions or 'none'}")
print()
for dimension, contribution in sorted(explanation.contributions.items(), key=lambda kv: -kv[1]):
    value = explanation.dimension_values[dimension]
    shown = f"{value:.3f}" if value is not None else "missing"
    print(f"  {dimension:20s} raw={shown:>8s}  contributes {contribution:.4f}")
print(f"\n  {'total':20s} {sum(explanation.contributions.values()):.4f}")

## 5. How much of this is publication bias?

Context.md §32.2 requires measuring the ranking with and without literature evidence.
This is the most important cell in the notebook.

In [ ]:
movement = ablation_movement(result, top_n=20).sort("rank_change")
movement.head(8)

In [ ]:
survivors = result.known_genes_in_top_20_without_literature
print(
    f"Known genes in top 20 WITH literature:    {len(result.known_genes_in_top_20)}/5 — {result.known_genes_in_top_20}"
)
print(f"Known genes in top 20 WITHOUT literature: {len(survivors)}/5 — {survivors}")

**Read this carefully.** PINK1 and PRKN fall out of the top 20 once publication
evidence is removed, so the headline result is partly literature-dependent.

That does not make literature worthless — PINK1 and PRKN genuinely matter. It means
this method cannot separate *well-published because important* from *merely often
mentioned*. Doing so needs labels and a held-out evaluation: Milestone 2.

## 6. Evidence breakdown figure

Stacked bars because contributions sum to the score — the stack total *is* the bar length.

In [ ]:
from IPython.display import Image

from target_prioritization.milestone1 import FIGURE_NAME
from target_prioritization.utils.paths import FIGURES_DIR

Image(filename=str(FIGURES_DIR / FIGURE_NAME))

## 7. Limitations

See `reports/parkinsons_baseline_report.md` section 6 for the full list. The three that
most constrain how this output should be read:

1. **The weights are arbitrary** — hand-set, tuned against no objective.
2. **This cannot discover anything new** — candidates are targets Open Targets *already*
   associates with Parkinson's, so the method re-ranks known associations (§13).
3. **No pathway, expression or network evidence** — three of the six §17.1 dimensions are
   absent, so a target whose case rests on those is under-scored here.